# Đo VRAM khi huấn luyện trên GPU Colab (T4)

Đo VRAM đỉnh của một bước huấn luyện (forward + backward + Adam, batch 256) với đúng kiến trúc mạng của cấp 5/6 (6 khối) và cấp 8 (4 khối), đồng thời in tên GPU làm bằng chứng phần cứng cho báo cáo (đề bài mục 3.4, 3.5).

Cách chạy trong VS Code: **Select Kernel → Colab → New Colab Server → GPU (T4)**, rồi **Run All**. Runtime Colab không thấy file trên máy, nên ô đầu sẽ clone kho mã về.

In [1]:
%cd /content
!test -d TTNTNC-NHOM31 || git clone https://github.com/minhhieu310800-jpg/TTNTNC-NHOM31.git
%cd /content/TTNTNC-NHOM31/do-an-caro

/content
Cloning into 'TTNTNC-NHOM31'...
remote: Enumerating objects: 851, done.
remote: Counting objects: 100% (389/389), done.
remote: Compressing objects: 100% (225/225), done.
remote: Total 851 (delta 190), reused 261 (delta 163), pack-reused 462 (from 1)
Receiving objects: 100% (851/851), 23.54 MiB | 24.57 MiB/s, done.
Resolving deltas: 100% (364/364), done.
/content/TTNTNC-NHOM31/do-an-caro


In [2]:
import json, platform
import torch, torch.nn.functional as F
from agents.network import PolicyValueNet

assert torch.cuda.is_available(), 'Chưa chọn runtime GPU: Select Kernel -> Colab -> GPU (T4)'
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu, '| torch', torch.__version__, '| CUDA', torch.version.cuda)

rows = []
for name, blocks, batch in [('cap5_6', 6, 256), ('cap8', 4, 256)]:
    torch.manual_seed(0)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    net = PolicyValueNet(blocks=blocks, channels=64, size=11).cuda()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
    x = (torch.rand(batch, 4, 11, 11, device='cuda') < 0.2).float()
    p = torch.randint(0, 121, (batch,), device='cuda')
    v = torch.rand(batch, device='cuda') * 2 - 1
    for _ in range(20):
        logits, value = net(x)
        loss = F.cross_entropy(logits, p) + F.mse_loss(value.view(-1), v)
        opt.zero_grad(); loss.backward(); opt.step()
    torch.cuda.synchronize()
    row = {'network': name, 'blocks': blocks, 'batch': batch,
           'params': sum(q.numel() for q in net.parameters()),
           'peak_allocated_mb': round(torch.cuda.max_memory_allocated() / 2**20, 1),
           'peak_reserved_mb': round(torch.cuda.max_memory_reserved() / 2**20, 1)}
    rows.append(row)
    print(f"{name}: {row['params']:,} tham số | VRAM đỉnh {row['peak_allocated_mb']} MB "
          f"(reserved {row['peak_reserved_mb']} MB, batch {batch})")

print(json.dumps({'gpu': gpu, 'torch': torch.__version__, 'cuda': torch.version.cuda,
                  'platform': platform.platform(), 'rows': rows}, ensure_ascii=False))

GPU: Tesla T4 | torch 2.11.0+cu130 | CUDA 13.0
cap5_6: 483,810 tham số | VRAM đỉnh 381.8 MB (reserved 514.0 MB, batch 256)
cap8: 335,842 tham số | VRAM đỉnh 319.6 MB (reserved 474.0 MB, batch 256)
{"gpu": "Tesla T4", "torch": "2.11.0+cu130", "cuda": "13.0", "platform": "Linux-6.6.159+-x86_64-with-glibc2.39", "rows": [{"network": "cap5_6", "blocks": 6, "batch": 256, "params": 483810, "peak_allocated_mb": 381.8, "peak_reserved_mb": 514.0}, {"network": "cap8", "blocks": 4, "batch": 256, "params": 335842, "peak_allocated_mb": 319.6, "peak_reserved_mb": 474.0}]}
